# Parsing concatenated address field
This code uses techniques such as regex to parse the column 'Town - Development - Address' into separate columns.

---

### Datasets
`Resale Transaction Info - Confidential_updated Sept 2025.csv` (https://drive.google.com/file/d/1EQdwdj5d6qpwnxoxfL0bbvhCG-CaJxUm/view?usp=drive_link)

---
### Instructions
Instruction: After cloning our repo, download the CSV files above and move it to the ds-chapa-affordable-housing/fa25-team-a/data folder, then locally run the cells below

---

The following is the pipeline:

1.   Splitting concatenated components using \<br>
2.   Extract Town and Development, by using the hyphen (-) to split
3. Renaming the new columns
4. Extracting the unit number
5. Dropping the original concatenated column
6. Downloading the parsed CSV.

In [ ]:
import pandas as pd

# --- Step 1: Load dataset ---
# Note: The dataset may have been exported from Excel with a long name;
# adjust the file path as needed for your environment.

df = pd.read_csv(
    "../../data/Resale Transaction Info - Confidential_updated Sept 2025.csv",
    dtype=str
)

# df = pd.read_csv(
#     "../data/Resale Transaction Info - Confidential_updated Sept 2025.csv",
#     dtype=str
# )

# --- Step 2: Split the concatenated string by HTML line breaks (<br>) ---
# Each cell contains multiple components separated by "</br>"
split_df = df['Town - Development - Address'].str.split('</br>', expand=True)

# --- Step 3: Extract Town and Development from the first part ---
# The format is typically: "Town - Development"
split_df[['Town', 'Development']] = split_df[0].str.split(' - ', expand=True)

# --- Step 4: Clean and assign extracted values back to the main DataFrame ---
df['Town'] = split_df['Town'].str.strip()
df['Development'] = split_df['Development'].str.strip()

# The second part often contains address details (e.g., street name)
# Remove any embedded "Unit:" text for clarity.
df['Address'] = (
    split_df[1]
    .str.strip()
    .str.replace(r'Unit:.*', '', regex=True)  # Remove "Unit:" and following text
    .str.strip()
)

# --- Step 5: Extract unit number (if present) ---
# Sometimes unit numbers appear in the third split segment.
df['Unit Number'] = (
    split_df[2]
    .str.replace('Unit:', '', regex=False)
    .str.replace('Unit', '', regex=False)
    .str.strip()
)

# --- Step 6: Optional cleanup ---
# Uncomment the line below to replace empty unit entries with NaN values
# df['Unit Number'].replace('', pd.NA, inplace=True)

# --- Step 7: Drop the original concatenated column ---
df.drop(columns=['Town - Development - Address'], inplace=True)

# --- Step 8: Save and export the cleaned dataset ---
output_path = "parsed_addresses.csv"
df.to_csv(output_path, index=False)

print(f"✅ Parsed dataset successfully saved as: {output_path}")

# --- Step 9: Preview the cleaned data ---
df.head()

✅ Parsed dataset successfully saved as: parsed_addresses.csv


,Age Restricted,Bedrooms,Transaction Start Date,Closing Date,Full/Lite Documentation,Town,Development,Address,Unit Number
0,55+,2,1/31/2021,9/30/2022,Full,Pembroke,Barker Square,7 Barker Square Drive,23
1,NaN,3,3/11/2021,7/7/2021,Full,Great Barrington,Blue Hill Commons,6 Emily Court,
2,NaN,3,5/17/2021,9/17/2021,Full,Taunton,Powhattan Estates,65 Metacomet Avenue,
3,NaN,2,5/19/2021,8/16/2021,Full,Bedford,The Village at Bedford Woods,1301 Albion Road,1301
4,NaN,2,7/12/2021,10/28/2021,Full,Billerica,Barrett Farm,41 Boston Road,233
